<a target="_blank" href="https://colab.research.google.com/github/Tensor-Reloaded/Neural-Networks-Template-2025/blob/main/Lab02/Assignment1.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# **Assignment 1 (10 points)**

## **Solving a linear system in python**

In this homework, you will familiarize yourself with key linear algebra con-
cepts and Python programming by solving a system of linear equations. You
will explore multiple methods for solving such systems, including Cramer’s rule
and matrix inversion. By the end of this assignment, you will have a good un-
derstanding of how to represent and manipulate matrices and vectors in Python.

We begin with the following system of 3 linear equations with 3 unknowns:
$$ 2x + 3y - z = 5 $$
$$ x - y + 4z = 6 $$
$$ 3x + y + 2z = 7 $$

This system can be vectorized in the following form:
$$ A \cdot X = B $$
where:
$$
A = \begin{bmatrix}
2 & 3 & -1 \\
1 & -1 & 4 \\
3 & 1 & 2
\end{bmatrix}, \quad 
X = \begin{bmatrix}
x \\
y \\
z
\end{bmatrix}, \quad 
B = \begin{bmatrix}
5 \\
6 \\
7
\end{bmatrix}
$$

**Considerations**
- do not use any linear algebra framework such as $numpy$
- use python lists as data structures for matrices and vectors
- experiment with other values for the coefficients and free terms

### **1. Parsing the System of Equations (1 point)**

The first task is to implement a Python script that reads a system of linear equations from a text file and parses it into a matrix $A$ and a vector $B$. You will use the input format described below to extract the coefficients for $A$ and $B$.

**Input File Format**
```text
2x + 3y - z = 5
x - y + 4z = 6
3x + y + 2z = 7
```

Note that the coefficients are always in the order x, y and z and the terms are always space separated

In [1]:
import pathlib
from fractions import Fraction

# ---------- mici ajutoare pentru afișare ----------
def fmt(value: float) -> str:
    """Număr frumos: 2.0 -> '2', 0.357142... -> '0.3571'."""
    value = 0.0 if abs(value) < 1e-12 else value
    return f"{value:.4f}".rstrip("0").rstrip(".")

def frac(value: float) -> str:
    """Fracția cea mai apropiată, de ex. 0.357142... -> '5/14'."""
    return str(Fraction(value).limit_denominator(1000))

def print_matrix(name: str, matrix: list[list[float]], indent: str = "    ") -> None:
    cells = [[fmt(v) for v in row] for row in matrix]
    width = max(len(c) for row in cells for c in row)
    print(f"{indent}{name} =")
    for row in cells:
        print(f"{indent}  [ " + "  ".join(c.rjust(width) for c in row) + " ]")

def print_vector(name: str, vector: list[float], indent: str = "    ") -> None:
    print(f"{indent}{name} = [" + ", ".join(fmt(v) for v in vector) + "]")


# ---------- 1. Parsing ----------
def parse_coefficient(token: str) -> float:
    """Transformă un termen ca '2x', 'x', '-z' sau '4.5y' în coeficientul lui."""
    number = token[:-1]  # scoatem litera variabilei
    if number in ("", "+"):
        return 1.0
    if number == "-":
        return -1.0
    return float(number)

def load_system(path: pathlib.Path, verbose: bool = False) -> tuple[list[list[float]], list[float]]:
    A = []
    B = []
    variables = ["x", "y", "z"]
    for line_number, line in enumerate(path.read_text().splitlines(), start=1):
        if not line.strip():
            continue
        left, right = line.split("=")
        coefficients = {v: 0.0 for v in variables}  # o variabilă care lipsește are coeficientul 0
        sign = 1.0
        if verbose:
            print(f"Ecuația {line_number}:  {line}")
            print(f"    tokeni (separați prin spații): {left.split()}  |  rezultat: {right.strip()}")
        for token in left.split():
            if token == "+":
                sign = 1.0
            elif token == "-":
                sign = -1.0
            else:
                value = sign * parse_coefficient(token)
                coefficients[token[-1]] = value
                if verbose:
                    print(f"    '{token}' -> variabila {token[-1]}, coeficient {fmt(value)}")
                sign = 1.0
        A.append([coefficients[v] for v in variables])
        B.append(float(right.strip()))
        if verbose:
            print(f"    => rândul {line_number} din A = {[fmt(c) for c in A[-1]]},  B[{line_number}] = {fmt(B[-1])}\n")
    return A, B

print("=== 1. Citim sistemul din system.txt ===\n")
A, B = load_system(pathlib.Path("system.txt"), verbose=True)
print("Rezultat:")
print_matrix("A (coeficienții)", A)
print_vector("B (termenii liberi)", B)
print(f"\n{A=} {B=}")

=== 1. Citim sistemul din system.txt ===

Ecuația 1:  2x + 3y - z = 5
    tokeni (separați prin spații): ['2x', '+', '3y', '-', 'z']  |  rezultat: 5
    '2x' -> variabila x, coeficient 2
    '3y' -> variabila y, coeficient 3
    'z' -> variabila z, coeficient -1
    => rândul 1 din A = ['2', '3', '-1'],  B[1] = 5

Ecuația 2:  x - y + 4z = 6
    tokeni (separați prin spații): ['x', '-', 'y', '+', '4z']  |  rezultat: 6
    'x' -> variabila x, coeficient 1
    'y' -> variabila y, coeficient -1
    '4z' -> variabila z, coeficient 4
    => rândul 2 din A = ['1', '-1', '4'],  B[2] = 6

Ecuația 3:  3x + y + 2z = 7
    tokeni (separați prin spații): ['3x', '+', 'y', '+', '2z']  |  rezultat: 7
    '3x' -> variabila x, coeficient 3
    'y' -> variabila y, coeficient 1
    '2z' -> variabila z, coeficient 2
    => rândul 3 din A = ['3', '1', '2'],  B[3] = 7

Rezultat:
    A (coeficienții) =
      [  2   3  -1 ]
      [  1  -1   4 ]
      [  3   1   2 ]
    B (termenii liberi) = [5, 6, 7]

A=[[2.0,

### **2. Matrix and Vector Operations (5 points)**

Once you have successfully parsed the matrix and vector, complete the following exercises to manipulate and understand basic matrix and vector operations. Write Python functions for each of these tasks:

#### 2.1. Determinant

Write a function to compute the determinant of matrix $A$. Recall one of the formulae for the determinant of a $3x3$ matrix:
$$ \text{det}(A) = a_{11}(a_{22}a_{33} - a_{23}a_{32}) - a_{12}(a_{21}a_{33} - a_{23}a_{31}) + a_{13}(a_{21}a_{32} - a_{22}a_{31}) $$

In [2]:
def determinant(matrix: list[list[float]]) -> float:
    if len(matrix) == 2:
        return matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0]
    (a11, a12, a13), (a21, a22, a23), (a31, a32, a33) = matrix
    return (a11 * (a22 * a33 - a23 * a32)
            - a12 * (a21 * a33 - a23 * a31)
            + a13 * (a21 * a32 - a22 * a31))

def explain_determinant(matrix: list[list[float]], name: str = "A") -> float:
    """Afișează dezvoltarea după primul rând: + a11·det(M11) − a12·det(M12) + a13·det(M13)."""
    print(f"det({name}) = + a11·det(M11) − a12·det(M12) + a13·det(M13)")
    signs = ["+", "−", "+"]
    total = 0.0
    for j in range(3):
        m = [[matrix[r][c] for c in range(3) if c != j] for r in (1, 2)]  # tăiem rândul 1 și coloana j
        a, b, c, d = m[0][0], m[0][1], m[1][0], m[1][1]
        det_m = a * d - b * c
        term = (1 if j != 1 else -1) * matrix[0][j] * det_m
        total += term
        print(f"  termen {j + 1}: tăiem rândul 1 și coloana {j + 1} -> M1{j + 1} = [[{fmt(a)}, {fmt(b)}], [{fmt(c)}, {fmt(d)}]]")
        print(f"           det(M1{j + 1}) = ({fmt(a)})·({fmt(d)}) − ({fmt(b)})·({fmt(c)}) = {fmt(a * d)} − ({fmt(b * c)}) = {fmt(det_m)}")
        print(f"           {signs[j]} ({fmt(matrix[0][j])}) × ({fmt(det_m)}) = {fmt(term)}")
    print(f"  det({name}) = suma termenilor = {fmt(total)}")
    return total

print("=== 2.1 Determinantul lui A ===\n")
print_matrix("A", A)
print()
explain_determinant(A)
print("\ndet(A) ≠ 0  =>  sistemul are o soluție unică" if determinant(A) != 0 else "\ndet(A) = 0  =>  nu există soluție unică")
print(f"\n{determinant(A)=}")

=== 2.1 Determinantul lui A ===

    A =
      [  2   3  -1 ]
      [  1  -1   4 ]
      [  3   1   2 ]

det(A) = + a11·det(M11) − a12·det(M12) + a13·det(M13)
  termen 1: tăiem rândul 1 și coloana 1 -> M11 = [[-1, 4], [1, 2]]
           det(M11) = (-1)·(2) − (4)·(1) = -2 − (4) = -6
           + (2) × (-6) = -12
  termen 2: tăiem rândul 1 și coloana 2 -> M12 = [[1, 4], [3, 2]]
           det(M12) = (1)·(2) − (4)·(3) = 2 − (12) = -10
           − (3) × (-10) = 30
  termen 3: tăiem rândul 1 și coloana 3 -> M13 = [[1, -1], [3, 1]]
           det(M13) = (1)·(1) − (-1)·(3) = 1 − (-3) = 4
           + (-1) × (4) = -4
  det(A) = suma termenilor = 14

det(A) ≠ 0  =>  sistemul are o soluție unică

determinant(A)=14.0


#### 2.2. Trace

Compute the sum of the elements along the main diagonal of matrix $A$. For a matrix $A$, this is:
$$ \text{Trace}(A) = a_{11} + a_{22} + a_{33} $$

In [3]:
def trace(matrix: list[list[float]]) -> float:
    return sum(matrix[i][i] for i in range(len(matrix)))

print("=== 2.2 Urma (trace) = suma diagonalei principale ===\n")
diagonal = [A[i][i] for i in range(len(A))]
print(f"diagonala principală: a11, a22, a33 = {', '.join(fmt(v) for v in diagonal)}")
print("trace(A) = " + " + ".join(f"({fmt(v)})" for v in diagonal) + f" = {fmt(trace(A))}")
print(f"\n{trace(A)=}")

=== 2.2 Urma (trace) = suma diagonalei principale ===

diagonala principală: a11, a22, a33 = 2, -1, 2
trace(A) = (2) + (-1) + (2) = 3

trace(A)=3.0


#### 2.3. Vector norm

Compute the Euclidean norm of vector $B$, which is:
$$ ||B|| = \sqrt{b_1^2 + b_2^2 + b_3^2} $$

In [4]:
import math

def norm(vector: list[float]) -> float:
    return math.sqrt(sum(b * b for b in vector))

print("=== 2.3 Norma euclidiană (lungimea vectorului B) ===\n")
squares = [b * b for b in B]
print("pătratele: " + ", ".join(f"{fmt(b)}² = {fmt(s)}" for b, s in zip(B, squares)))
print(f"suma pătratelor = {' + '.join(fmt(s) for s in squares)} = {fmt(sum(squares))}")
print(f"||B|| = √{fmt(sum(squares))} = {fmt(norm(B))}")
print(f"\n{norm(B)=}")

=== 2.3 Norma euclidiană (lungimea vectorului B) ===

pătratele: 5² = 25, 6² = 36, 7² = 49
suma pătratelor = 25 + 36 + 49 = 110
||B|| = √110 = 10.4881

norm(B)=10.488088481701515


#### 2.4. Transpose of matrix

Write a function to compute the transpose of matrix $A$. The transpose of a matrix $A$ is obtained by swapping its rows and columns.
    

In [5]:
def transpose(matrix: list[list[float]]) -> list[list[float]]:
    rows, cols = len(matrix), len(matrix[0])
    return [[matrix[i][j] for i in range(rows)] for j in range(cols)]

print("=== 2.4 Transpusa: rândurile devin coloane ===\n")
print_matrix("A", A)
for i, row in enumerate(A, start=1):
    print(f"    rândul {i} din A {[fmt(v) for v in row]} devine coloana {i} din Aᵀ")
print_matrix("Aᵀ", transpose(A))
print(f"\n{transpose(A)=}")

=== 2.4 Transpusa: rândurile devin coloane ===

    A =
      [  2   3  -1 ]
      [  1  -1   4 ]
      [  3   1   2 ]
    rândul 1 din A ['2', '3', '-1'] devine coloana 1 din Aᵀ
    rândul 2 din A ['1', '-1', '4'] devine coloana 2 din Aᵀ
    rândul 3 din A ['3', '1', '2'] devine coloana 3 din Aᵀ
    Aᵀ =
      [  2   1   3 ]
      [  3  -1   1 ]
      [ -1   4   2 ]

transpose(A)=[[2.0, 1.0, 3.0], [3.0, -1.0, 1.0], [-1.0, 4.0, 2.0]]


#### 2.5. Matrix-vector multiplication

Write a function that multiplies matrix $A$ with vector $B$.

In [6]:
def multiply(matrix: list[list[float]], vector: list[float]) -> list[float]:
    return [sum(row[j] * vector[j] for j in range(len(vector))) for row in matrix]

print("=== 2.5 Înmulțirea A · B: fiecare rând din A înmulțit cu B ===\n")
for i, row in enumerate(A, start=1):
    products = " + ".join(f"({fmt(row[j])})·({fmt(B[j])})" for j in range(len(B)))
    print(f"rândul {i}: {products} = {fmt(multiply([row], B)[0])}")
print_vector("A · B", multiply(A, B), indent="")
print(f"\n{multiply(A, B)=}")

=== 2.5 Înmulțirea A · B: fiecare rând din A înmulțit cu B ===

rândul 1: (2)·(5) + (3)·(6) + (-1)·(7) = 21
rândul 2: (1)·(5) + (-1)·(6) + (4)·(7) = 27
rândul 3: (3)·(5) + (1)·(6) + (2)·(7) = 35
A · B = [21, 27, 35]

multiply(A, B)=[21.0, 27.0, 35.0]


### **3. Solving using Cramer's Rule (1 point)**

Now that you have explored basic matrix operations, solve the system of linear equations using Cramer's rule.

**Cramer's Rule:**

Cramer's rule allows you to solve for each unknown $x$, $y$, and $z$ using determinants. For example:
$$ x = \frac{\text{det}(A_x)}{\text{det}(A)}, \quad y = \frac{\text{det}(A_y)}{\text{det}(A)}, \quad z = \frac{\text{det}(A_z)}{\text{det}(A)} $$
where $A_x$, $A_y$, and $A_z$ are matrices formed by replacing the respective column of matrix $A$ with vector $B$.

In [7]:
def replace_column(matrix: list[list[float]], column: int, vector: list[float]) -> list[list[float]]:
    return [[vector[i] if j == column else matrix[i][j] for j in range(len(matrix[i]))]
            for i in range(len(matrix))]

def solve_cramer(matrix: list[list[float]], vector: list[float]) -> list[float]:
    det_a = determinant(matrix)
    if det_a == 0:
        raise ValueError("det(A) = 0: the system has no unique solution")
    return [determinant(replace_column(matrix, k, vector)) / det_a for k in range(len(vector))]

print("=== 3. Regula lui Cramer: x = det(Ax)/det(A), y = det(Ay)/det(A), z = det(Az)/det(A) ===\n")
det_a = determinant(A)
print(f"det(A) = {fmt(det_a)}  (calculat la 2.1)\n")
for k, name in enumerate("xyz"):
    A_k = replace_column(A, k, B)
    print(f"--- {name}: punem B în locul coloanei {k + 1} (coloana lui {name}) ---")
    print_matrix(f"A{name}", A_k)
    det_k = explain_determinant(A_k, f"A{name}")
    print(f"  {name} = det(A{name}) / det(A) = {fmt(det_k)} / {fmt(det_a)} = {fmt(det_k / det_a)}  (= {frac(det_k / det_a)})\n")
print(f"{solve_cramer(A, B)=}")

=== 3. Regula lui Cramer: x = det(Ax)/det(A), y = det(Ay)/det(A), z = det(Az)/det(A) ===

det(A) = 14  (calculat la 2.1)

--- x: punem B în locul coloanei 1 (coloana lui x) ---
    Ax =
      [  5   3  -1 ]
      [  6  -1   4 ]
      [  7   1   2 ]
det(Ax) = + a11·det(M11) − a12·det(M12) + a13·det(M13)
  termen 1: tăiem rândul 1 și coloana 1 -> M11 = [[-1, 4], [1, 2]]
           det(M11) = (-1)·(2) − (4)·(1) = -2 − (4) = -6
           + (5) × (-6) = -30
  termen 2: tăiem rândul 1 și coloana 2 -> M12 = [[6, 4], [7, 2]]
           det(M12) = (6)·(2) − (4)·(7) = 12 − (28) = -16
           − (3) × (-16) = 48
  termen 3: tăiem rândul 1 și coloana 3 -> M13 = [[6, -1], [7, 1]]
           det(M13) = (6)·(1) − (-1)·(7) = 6 − (-7) = 13
           + (-1) × (13) = -13
  det(Ax) = suma termenilor = 5
  x = det(Ax) / det(A) = 5 / 14 = 0.3571  (= 5/14)

--- y: punem B în locul coloanei 2 (coloana lui y) ---
    Ay =
      [  2   5  -1 ]
      [  1   6   4 ]
      [  3   7   2 ]
det(Ay) = + a11·det(M1

### **4. Solving using Inversion (3 points)**

Finally, solve the system by computing the inverse of matrix $A$ and multiplying it by vector $B$.
$$ A \cdot X = B \rightarrow X = A^{-1} \cdot B $$
**Adjugate Method for Matrix Inversion:**

To find the inverse of matrix $ A $, you can use the adjugate method:
$$ A^{-1} = \frac{1}{\text{det}(A)} \times \text{adj}(A) $$
where $\text{adj}(A)$ is the adjugate (or adjoint) matrix, which is the transpose of the cofactor matrix of $ A $.

**Cofactor Matrix:**

The cofactor matrix is a matrix where each element is replaced by its cofactor. The cofactor of an element $a_{ij}$ is given by:
$$ (-1)^{i+j} \times \text{det}(M_{ij}) $$
where $M_{ij}$ is the minor of element $a_{ij}$, which is the matrix obtained by removing the $i$-th row and $j$-th column from matrix $A$.

In [8]:
def minor(matrix: list[list[float]], i: int, j: int) -> list[list[float]]:
    return [[matrix[r][c] for c in range(len(matrix)) if c != j]
            for r in range(len(matrix)) if r != i]

def cofactor(matrix: list[list[float]]) -> list[list[float]]:
    n = len(matrix)
    return [[(-1) ** (i + j) * determinant(minor(matrix, i, j)) for j in range(n)]
            for i in range(n)]

def adjoint(matrix: list[list[float]]) -> list[list[float]]:
    return transpose(cofactor(matrix))

def solve(matrix: list[list[float]], vector: list[float]) -> list[float]:
    det_a = determinant(matrix)
    if det_a == 0:
        raise ValueError("det(A) = 0: the matrix is not invertible")
    inverse = [[value / det_a for value in row] for row in adjoint(matrix)]
    return multiply(inverse, vector)

print("=== 4. Rezolvare cu inversa: A·X = B  =>  X = A⁻¹·B,  A⁻¹ = adj(A) / det(A) ===\n")

print("Pasul 1: matricea cofactorilor C, celulă cu celulă")
print("         C[i][j] = (−1)^(i+j) × det(minorul fără rândul i și coloana j)\n")
for i in range(3):
    for j in range(3):
        m = minor(A, i, j)
        a, b, c, d = m[0][0], m[0][1], m[1][0], m[1][1]
        det_m = determinant(m)
        sign = "+" if (i + j) % 2 == 0 else "−"
        print(f"  C{i + 1}{j + 1}: tăiem rândul {i + 1} și coloana {j + 1} -> [[{fmt(a)}, {fmt(b)}], [{fmt(c)}, {fmt(d)}]]"
              f",  det = ({fmt(a)})·({fmt(d)}) − ({fmt(b)})·({fmt(c)}) = {fmt(det_m)}"
              f",  semn {sign}  ->  C{i + 1}{j + 1} = {fmt((-1) ** (i + j) * det_m)}")
C = cofactor(A)
print()
print_matrix("C", C)

print("\nPasul 2: adjuncta = transpusa lui C (rândurile lui C devin coloane)")
adj = adjoint(A)
print_matrix("adj(A)", adj)

det_a = determinant(A)
inverse = [[value / det_a for value in row] for row in adj]
print(f"\nPasul 3: A⁻¹ = adj(A) / det(A) = adj(A) / {fmt(det_a)}  (fiecare număr împărțit la {fmt(det_a)})")
print_matrix("A⁻¹", inverse)
print("    ca fracții: " + "  ".join("[" + ", ".join(frac(v) for v in row) + "]" for row in inverse))

identity = [[sum(inverse[i][k] * A[k][j] for k in range(3)) for j in range(3)] for i in range(3)]
print("\nVerificare: A⁻¹ · A trebuie să dea matricea identitate I")
print_matrix("A⁻¹ · A", identity)

print("\nPasul 4: X = A⁻¹ · B = (adj(A) · B) / det(A)  (fiecare rând din adj(A) înmulțit cu B, apoi împărțit)")
for r, name in enumerate("xyz"):
    products = " + ".join(f"({fmt(adj[r][j])})·({fmt(B[j])})" for j in range(3))
    value = multiply([inverse[r]], B)[0]
    print(f"  {name} = [{products}] / {fmt(det_a)} = {fmt(sum(adj[r][j] * B[j] for j in range(3)))} / {fmt(det_a)} = {fmt(value)}  (= {frac(value)})")

X = solve(A, B)
print("\nComparare cu Cramer:", "aceleași valori ✓" if all(abs(p - q) < 1e-9 for p, q in zip(X, solve_cramer(A, B))) else "diferă!")
print_vector("A · X (trebuie să fie B)", multiply(A, X), indent="")
print(f"\n{solve(A, B)=}")

=== 4. Rezolvare cu inversa: A·X = B  =>  X = A⁻¹·B,  A⁻¹ = adj(A) / det(A) ===

Pasul 1: matricea cofactorilor C, celulă cu celulă
         C[i][j] = (−1)^(i+j) × det(minorul fără rândul i și coloana j)

  C11: tăiem rândul 1 și coloana 1 -> [[-1, 4], [1, 2]],  det = (-1)·(2) − (4)·(1) = -6,  semn +  ->  C11 = -6
  C12: tăiem rândul 1 și coloana 2 -> [[1, 4], [3, 2]],  det = (1)·(2) − (4)·(3) = -10,  semn −  ->  C12 = 10
  C13: tăiem rândul 1 și coloana 3 -> [[1, -1], [3, 1]],  det = (1)·(1) − (-1)·(3) = 4,  semn +  ->  C13 = 4
  C21: tăiem rândul 2 și coloana 1 -> [[3, -1], [1, 2]],  det = (3)·(2) − (-1)·(1) = 7,  semn −  ->  C21 = -7
  C22: tăiem rândul 2 și coloana 2 -> [[2, -1], [3, 2]],  det = (2)·(2) − (-1)·(3) = 7,  semn +  ->  C22 = 7
  C23: tăiem rândul 2 și coloana 3 -> [[2, 3], [3, 1]],  det = (2)·(1) − (3)·(3) = -7,  semn −  ->  C23 = 7
  C31: tăiem rândul 3 și coloana 1 -> [[3, -1], [-1, 4]],  det = (3)·(4) − (-1)·(-1) = 11,  semn +  ->  C31 = 11
  C32: tăiem rândul 3 și 